In [ ]:
import tensorflow as tf
# Allocate GPU memory only as needed. By default TensorFlow grabs all of it, which makes a second
# notebook (or kernel) fail with an out-of-memory error on the 4 GB laptop GPU.
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from tensorflow import keras

In [ ]:
X, y = make_moons(n_samples=100, noise=0.2, random_state=42)
X[0].shape

In [ ]:
def build_model(rate):
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(2,)),
        keras.layers.Dropout(rate),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(rate),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:
for rate in [0.0, 0.2, 0.5]:
    model = build_model(rate)
    history = model.fit(X, y, epochs=100, verbose=0)
    print('Dropout:', rate,
          'Accuracy:', model.evaluate(X, y, verbose=0)[1])
    plt.plot(history.history['accuracy'])
    plt.title('Training Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.show()

In [ ]:

!pip install mlxtend

import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l1, l2

X, y = make_moons(100, noise=0.25, random_state=2)

def build_model(kind):
    reg = l1(0.001) if kind == 'L1' else l2(0.001) if kind == 'L2' else None

    model = keras.Sequential([
        Dense(128, activation='relu', input_shape=(2,), kernel_regularizer=reg),
        Dense(128, activation='relu', kernel_regularizer=reg)
    ])

    if kind == 'Dropout':
        model.add(Dropout(0.5))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=keras.optimizers.Adam(0.01),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

for kind in ['None', 'L1', 'L2', 'Dropout']:
    model = build_model(kind)
    h = model.fit(X, y, epochs=500, validation_split=0.2, verbose=0)
    print(kind, 'Train:', h.history['accuracy'][-1],
          'Validation:', h.history['val_accuracy'][-1])

    plt.plot(h.history['loss'], label='Train')
    plt.plot(h.history['val_loss'], label='Validation')
    plt.title(kind)
    plt.legend()
    plt.show()
